# Hallucination Early-Warning Prototype (RAG answers) - version 3

**What this notebook does, in simple words**

1. Takes answers that other AI models wrote from a document (the RAGTruth dataset, with human marks on the wrong parts).
2. For each sentence it measures warning signs:
   - **A. Word-level doubt** - how unsure our model is on each word (one cheap pass).
   - **B. Meaning-level doubt (semantic entropy)** - our model re-writes the sentence 5 times. If the versions mean different things, it is guessing.
   - **C. Document check** - does the document back this sentence up? Now with **three outcomes**: supported / not mentioned / contradicted.
3. Mixes the signs into one risk score per sentence and compares with the human marks.
4. Tests the project's main question: **should the document check, or the word-level doubt, decide which sentences get the expensive re-writing?**
5. Also: early-warning test, whole-answer baseline (the paper's method), a safety rule for the demo, and a live demo.

**What is new in version 3**

- A paired test: does the expensive sampling really help? (section 6b)
- Global threshold vs one threshold per task (section 6c)
- A fairer baseline (SelfCheckGPT-style): the model re-writes the whole answer 5 times and we check whether those re-writes agree with the **written** answer, sentence by sentence (sections 5b and 10b).

**How to run**

1. `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.
2. Everything from version 2 is saved on Google Drive (`USE_DRIVE = True`), so the 200 answers and the first baseline are **not repeated**. Only the new SelfCheck-style cell does new heavy work (about 20 minutes).
3. If Colab disconnects, run all cells again and it carries on from the last save.

In [1]:
# Colab only. On your own machine use:  pip install -r requirements.txt
!pip -q install gradio accelerate scikit-learn pandas

In [2]:
# ---------- SETTINGS (change these if you want) ----------
import os, re, json, math, random, time, html, urllib.request
import numpy as np, pandas as pd, torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"     # the model that measures doubt (fits a free T4)
NLI_NAME   = "microsoft/deberta-large-mnli"  # a small model that checks "does A imply B?"
N_RESP     = 200    # real run (it resumes from saved answers; use 10 for a quick smoke test)
N_SAMPLES  = 5      # how many re-writes per sentence (signal B)
MAX_TOK    = 1500   # skip examples longer than this (keeps the T4 from running out of memory)
MIN_WORDS  = 4      # ignore tiny sentences like "Sure!"
SEED       = 42
RUN_BASELINE = True # also run the paper's whole-answer semantic entropy (needs extra time)
BASE_N     = 100    # how many answers get the whole-answer baselines (use 5 for a smoke test)
USE_DRIVE  = True   # save progress to Google Drive so a disconnect does not lose it

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if USE_DRIVE and IN_COLAB:                      # Colab + Google Drive: progress survives a disconnect
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/hallucination_project"
else:                                           # any other machine: plain local folders
    WORK = os.environ.get("HALLU_WORK", "./work")   # change with:  export HALLU_WORK=/some/folder
DATA = os.environ.get("HALLU_DATA", "./data")       # RAGTruth is downloaded here
os.makedirs(WORK, exist_ok=True); os.makedirs(DATA, exist_ok=True)

assert torch.cuda.is_available(), "A CUDA GPU is required. In Colab: Runtime > Change runtime type > T4 GPU"
dev = "cuda"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0), "| saving to:", WORK)

Mounted at /content/drive
GPU: Tesla T4 | saving to: /content/drive/MyDrive/hallucination_project


## 1. Load RAGTruth and pick a balanced sample
Half of the picked answers contain a human-marked hallucination, half are clean.

In [3]:
BASE_URL = "https://raw.githubusercontent.com/ParticleMedia/RAGTruth/main/dataset/"

def load_jsonl(name):
    path = f"{DATA}/{name}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE_URL + name, path)
    return [json.loads(l) for l in open(path, encoding="utf-8")]

def flat(x):
    if isinstance(x, str):  return x
    if isinstance(x, dict): return "\n".join(f"{k}: {flat(v)}" for k, v in x.items())
    if isinstance(x, list): return "\n".join(flat(v) for v in x)
    return str(x)

resp = load_jsonl("response.jsonl")
src  = {s["source_id"]: s for s in load_jsonl("source_info.jsonl")}

cand = [r for r in resp if r["split"] == "test" and r["quality"] == "good" and r["source_id"] in src]
hal  = [r for r in cand if r["labels"]]
cln  = [r for r in cand if not r["labels"]]
rng = random.Random(SEED); rng.shuffle(hal); rng.shuffle(cln)

order = []
for h, c in zip(hal, cln):
    for r in (h, c):
        s = src[r["source_id"]]
        source_text = s["source_info"]["passages"] if s["task_type"] == "QA" else flat(s["source_info"])
        order.append(dict(rid=r["id"], task=s["task_type"], writer=r["model"], prompt=s["prompt"],
                          source_text=source_text, response=r["response"], labels=r["labels"]))
print(len(order), "candidate answers ready. Tasks:", pd.Series([o["task"] for o in order]).value_counts().to_dict())

1886 candidate answers ready. Tasks: {'Data2txt': 747, 'Summary': 591, 'QA': 548}


## 2. Load the two helper models

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model = model.to(dev).eval()

nli_tok = AutoTokenizer.from_pretrained(NLI_NAME)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).half().to(dev).eval()
_lab = {i: n.upper() for i, n in nli.config.id2label.items()}
ENT = [i for i, n in _lab.items() if "ENTAIL" in n][0]
NEU = [i for i, n in _lab.items() if "NEUTRAL" in n][0]
CON = [i for i, n in _lab.items() if "CONTRA" in n][0]
print("models ready | NLI labels:", _lab)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

... (10 outputs left out)

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


models ready | NLI labels: {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}


## 3. The warning signs

All the work happens in `score_response`. The same function is used for the evaluation **and** for the live demo, so the demo is exactly what was measured.

New in this version: the document check keeps all **three** answers of the checking model (supported / contradicted / not mentioned), the cost of every re-write is recorded, and the word-level doubt of every word is saved for the early-warning test.

In [5]:
# ---------- text helpers ----------
_SPLIT = re.compile(r'(?<=[.!?])["\')\]]*\s+(?=[A-Z0-9"\'(\[])|\n+')

def split_sents(t):
    out, st = [], 0
    for m in _SPLIT.finditer(t):
        out.append((st, m.end())); st = m.end()
    if st < len(t): out.append((st, len(t)))
    return [(a, b) for a, b in out if t[a:b].strip()]

def first_sentence(t):
    t = t.strip()
    sp = split_sents(t)
    return t[sp[0][0]:sp[0][1]].strip() if sp else ""

def chat_prompt(user_text):
    return tok.apply_chat_template([{"role": "user", "content": user_text}],
                                   add_generation_prompt=True, tokenize=False)

def make_chunks(text, w=220, s=110, maxn=14):
    words = text.split()
    if not words: return []
    out = [" ".join(words[i:i + w]) for i in range(0, max(1, len(words) - w + s), s)]
    return out[:maxn]

# ---------- signal A: word-level doubt (teacher forcing, one pass) ----------
@torch.no_grad()
def token_stats(full_text, start_char):
    enc = tok(full_text, return_offsets_mapping=True, return_tensors="pt")
    ids = enc["input_ids"]
    if ids.shape[1] > MAX_TOK: return None
    offs = enc["offset_mapping"][0].tolist()
    i0 = next((i for i, o in enumerate(offs) if o[0] >= start_char), None)
    if i0 is None or i0 < 1 or i0 >= ids.shape[1]: return None
    logits = model(ids.to(dev), use_cache=False).logits[0, i0 - 1:-1].float()
    lp = torch.log_softmax(logits, -1)
    ent = -(lp.exp() * lp).sum(-1)
    nll = -lp.gather(1, ids[0, i0:].to(dev)[:, None])[:, 0]
    return [(offs[i0 + j][0] - start_char, offs[i0 + j][1] - start_char, float(ent[j]), float(nll[j]))
            for j in range(ids.shape[1] - i0)]

# ---------- NLI helpers (3 outputs: contradiction / neutral / entailment) ----------
@torch.no_grad()
def nli_probs(premises, hypotheses, bs=16):
    out = []
    for i in range(0, len(premises), bs):
        e = nli_tok(premises[i:i + bs], hypotheses[i:i + bs], return_tensors="pt",
                    padding=True, truncation=True, max_length=512).to(dev)
        out.append(nli(**e).logits.float().softmax(-1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 3))

def entail_probs(premises, hypotheses):
    return nli_probs(premises, hypotheses)[:, ENT].tolist()

# ---------- signal B: meaning-level doubt (semantic entropy) ----------
@torch.no_grad()
def sample_variants(prompt_txt, prefix, n_tok):
    text = prompt_txt + prefix.rstrip() if prefix.strip() else prompt_txt
    ids = tok(text, return_tensors="pt").to(dev)
    out = model.generate(**ids, do_sample=True, temperature=0.7, top_p=0.9, top_k=0,
                         num_return_sequences=N_SAMPLES,
                         max_new_tokens=min(70, int(1.4 * n_tok) + 8), pad_token_id=tok.eos_token_id)
    new = out[:, ids["input_ids"].shape[1]:]
    txt = tok.batch_decode(new, skip_special_tokens=True)
    res = [first_sentence(t) for t in txt]
    return [r for r in res if len(r.split()) >= 2], int(new.shape[0] * new.shape[1])   # 2nd value = cost (token slots)

def cluster_se(samples):
    # 0 = all re-writes mean the same, 1 = all different
    n = len(samples)
    if n < 2: return 0.0
    pairs = [(i, j) for i in range(n) for j in range(n) if i != j]
    pr = entail_probs([samples[i] for i, j in pairs], [samples[j] for i, j in pairs])
    P = dict(zip(pairs, pr))
    clusters = []
    for i in range(n):
        for c in clusters:
            if P[(c[0], i)] > 0.5 and P[(i, c[0])] > 0.5:
                c.append(i); break
        else:
            clusters.append([i])
    p = np.array([len(c) for c in clusters]) / n
    return float(max(0.0, -(p * np.log(p)).sum() / math.log(n)))

def se_and_disagreement(samples, written):
    n = len(samples)
    if n < 2: return 0.0, 0.5
    se = cluster_se(samples)
    dis = 1.0 - float(np.mean(entail_probs(samples, [written] * n)))   # do the re-writes agree with the written sentence?
    return se, dis

# ---------- signal C: does the document back the sentence up? (3 outcomes) ----------
def doc_scores(chunks, sentence):
    if not chunks: return dict(doc_ent=0.0, doc_con=0.0, doc_neu=1.0)
    P = nli_probs(chunks, [sentence] * len(chunks))
    ent, con = float(P[:, ENT].max()), float(P[:, CON].max())
    return dict(doc_ent=ent, doc_con=con, doc_neu=float(max(0.0, 1.0 - max(ent, con))))

# ---------- put it together ----------
def score_response(prompt, response, source_text, labels=None):
    ptxt = chat_prompt(prompt)
    toks = token_stats(ptxt + response, len(ptxt))
    if toks is None: return None
    chunks = make_chunks(source_text)
    rows = []
    for k, (a, b) in enumerate(split_sents(response)):
        s = response[a:b].strip()
        if len(s.split()) < MIN_WORDS: continue
        tt = [t for t in toks if t[0] < b and t[1] > a]
        if not tt: continue
        ent = np.array([t[2] for t in tt]); nll = np.array([t[3] for t in tt])
        variants, cost = sample_variants(ptxt, response[:a], len(tt))
        se, dis = se_and_disagreement(variants, s)
        d = doc_scores(chunks, s)
        lab = int(any(l["start"] < b and l["end"] > a for l in labels)) if labels is not None else None
        rows.append(dict(k=k, text=s, ent_mean=float(ent.mean()), ent_max=float(ent.max()),
                         nll_mean=float(nll.mean()), se=se, dis=dis, nosup=1.0 - d["doc_ent"],
                         doc_ent=d["doc_ent"], doc_con=d["doc_con"], doc_neu=d["doc_neu"],
                         gen_tok=cost, ntok=len(tt), label=lab))
    return (rows, toks) if rows else None

def span_info(toks, labels):
    # keep every word's doubt + where the human-marked spans are (used by the early-warning test)
    ent = [round(t[2], 3) for t in toks]
    spans = []
    for l in labels:
        idx = [i for i, t in enumerate(toks) if t[0] < l["end"] and t[1] > l["start"]]
        if idx: spans.append([min(idx), max(idx)])
    return dict(ent=ent, spans=spans)

## 4. Run on the sample (saves progress; safe to re-run)
For the real run, set `N_RESP = 200` in the settings cell and run this cell again - it continues where it stopped.

In [6]:
F_CSV, F_JSON = f"{WORK}/features_v2.csv", f"{WORK}/spans_v2.json"
ROWS, SPANS = [], []
if os.path.exists(F_CSV) and os.path.exists(F_JSON):
    ROWS = pd.read_csv(F_CSV).to_dict("records"); SPANS = json.load(open(F_JSON))
    print("resumed with", len(SPANS), "answers already done")
done = {str(s["rid"]) for s in SPANS}

def save():
    pd.DataFrame(ROWS).to_csv(F_CSV, index=False); json.dump(SPANS, open(F_JSON, "w"))

t0 = time.time(); start_n = len(done)
for r in order:
    if len(done) >= N_RESP: break
    if str(r["rid"]) in done: continue
    out = score_response(r["prompt"], r["response"], r["source_text"], r["labels"])
    if out is None: continue
    rows, toks = out
    for x in rows: x.update(rid=r["rid"], task=r["task"], writer=r["writer"])
    ROWS += rows
    sp = span_info(toks, r["labels"]); sp.update(rid=r["rid"], task=r["task"]); SPANS.append(sp)
    done.add(str(r["rid"]))
    if len(done) % 5 == 0 or len(done) == N_RESP:
        save(); torch.cuda.empty_cache()
        per = (time.time() - t0) / max(1, len(done) - start_n)
        print(f"{len(done)}/{N_RESP} answers | {len(ROWS)} sentences | ~{per*(N_RESP-len(done))/60:.0f} min left")
save(); print("done:", len(done), "answers,", len(ROWS), "sentences")

resumed with 200 answers already done
done: 200 answers, 1425 sentences


## 5. The paper's baseline: semantic entropy on the whole answer
The model writes 5 full answers to the same question. If they mean different things, the model is unsure about the whole answer. This is the method the MASE paper compares against. It only gives **one score per answer**, so it is compared with our method at answer level further below.
It also costs more: every re-write is a full answer, not one sentence.

In [7]:
B_CSV = f"{WORK}/baseline_v2.csv"
BASE = pd.read_csv(B_CSV).to_dict("records") if os.path.exists(B_CSV) else []
bdone = {str(b["rid"]) for b in BASE}
by_rid = {str(r["rid"]): r for r in order}

@torch.no_grad()
def whole_answer_se(prompt):
    ids = tok(chat_prompt(prompt), return_tensors="pt").to(dev)
    out = model.generate(**ids, do_sample=True, temperature=0.7, top_p=0.9, top_k=0,
                         num_return_sequences=N_SAMPLES, max_new_tokens=256, pad_token_id=tok.eos_token_id)
    new = out[:, ids["input_ids"].shape[1]:]
    txt = [t.strip() for t in tok.batch_decode(new, skip_special_tokens=True)]
    txt = [t for t in txt if len(t.split()) >= 2]
    return cluster_se(txt), int(new.shape[0] * new.shape[1])

if RUN_BASELINE:
    t0 = time.time(); todo = [str(s["rid"]) for s in SPANS][:BASE_N]
    for i, rid in enumerate(todo):
        if rid in bdone: continue
        r = by_rid[rid]
        se_w, cost = whole_answer_se(r["prompt"])
        BASE.append(dict(rid=rid, task=r["task"], se_whole=se_w, gen_tok=cost, label=int(bool(r["labels"]))))
        bdone.add(rid)
        if len(BASE) % 5 == 0:
            pd.DataFrame(BASE).to_csv(B_CSV, index=False); torch.cuda.empty_cache()
            print(f"baseline {len(BASE)}/{len(todo)} | {(time.time()-t0)/60:.1f} min so far")
    pd.DataFrame(BASE).to_csv(B_CSV, index=False)
    print("baseline done for", len(BASE), "answers")
else:
    print("RUN_BASELINE is False - skipped")

baseline done for 100 answers


## 5b. A fairer baseline: SelfCheckGPT-style check of the **written** answer
The whole-answer entropy baseline above never looks at the answer that was actually written (RAGTruth answers come from other AI models).
This version does: the model writes 5 full answers, and for every sentence of the **written** answer we ask whether those 5 answers contradict it.
If the model's own re-writes keep contradicting a sentence, that sentence is suspicious. This is the idea behind SelfCheckGPT, the baseline the MASE paper also compares with.
It costs about 20 minutes and resumes if interrupted.

In [8]:
SC_CSV = f"{WORK}/selfcheck_v2.csv"
SC = pd.read_csv(SC_CSV).to_dict("records") if os.path.exists(SC_CSV) else []
sc_done = {str(r["rid"]) for r in SC}

@torch.no_grad()
def gen_samples(prompt):
    ids = tok(chat_prompt(prompt), return_tensors="pt").to(dev)
    out = model.generate(**ids, do_sample=True, temperature=0.7, top_p=0.9, top_k=0,
                         num_return_sequences=N_SAMPLES, max_new_tokens=256, pad_token_id=tok.eos_token_id)
    new = out[:, ids["input_ids"].shape[1]:]
    txt = [t.strip() for t in tok.batch_decode(new, skip_special_tokens=True)]
    return [t for t in txt if len(t.split()) >= 2], int(new.shape[0] * new.shape[1])

def selfcheck_scores(response, samples):
    rows = []
    for k, (a, b) in enumerate(split_sents(response)):
        s_ = response[a:b].strip()
        if len(s_.split()) < MIN_WORDS: continue
        if not samples: rows.append((k, 0.5)); continue
        P = nli_probs(samples, [s_] * len(samples)); c, e = P[:, CON], P[:, ENT]
        rows.append((k, float(np.mean(c / (c + e + 1e-9)))))
    return rows

# ---- run ----
if RUN_BASELINE:
    t0 = time.time(); todo = [str(s_["rid"]) for s_ in SPANS][:BASE_N]; n_new = 0
    for rid in todo:
        if rid in sc_done: continue
        r = by_rid[rid]
        samples, cost = gen_samples(r["prompt"])
        for k, sc in selfcheck_scores(r["response"], samples):
            SC.append(dict(rid=rid, k=k, sc=sc, gen_tok=cost))
        sc_done.add(rid); n_new += 1
        if n_new % 5 == 0:
            pd.DataFrame(SC).to_csv(SC_CSV, index=False); torch.cuda.empty_cache()
            print(f"selfcheck {len(sc_done)}/{len(todo)} | {(time.time()-t0)/60:.1f} min so far")
    pd.DataFrame(SC).to_csv(SC_CSV, index=False)
    print("selfcheck-style scores ready for", len(sc_done), "answers")
else:
    print("RUN_BASELINE is False - skipped")

selfcheck-style scores ready for 100 answers


## 6. Results: does the combined warning beat each single sign?

Everything below is measured on sentences the model never saw labels for (5-fold split by answer, so no answer appears in both train and test).
The warning threshold is chosen **per task type** (QA / Summary / Data-to-text), using only training data.

`C3` is the new three-outcome document check (supported / contradicted / not mentioned).

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score,
                             recall_score, accuracy_score, brier_score_loss)

df = pd.DataFrame(ROWS).reset_index(drop=True)
df["label"] = df["label"].astype(int); df["rid"] = df["rid"].astype(str)
assert df.label.nunique() == 2, "No hallucinated sentences yet - raise N_RESP and run the run-cell again."
print(f"{len(df)} sentences from {df.rid.nunique()} answers | hallucinated sentences: {df.label.sum()} ({df.label.mean():.1%})")
display(df.groupby("task").label.agg(["count", "sum"]).rename(columns={"count": "sentences", "sum": "hallucinated"}).T)

class Const:
    def __init__(s, p): s.p = p
    def predict_proba(s, X): return np.column_stack([1 - np.full(len(X), s.p), np.full(len(X), s.p)])

def fit_lr(X, y):
    if len(set(y)) < 2: return Const(float(y.mean()))
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)).fit(X, y)

def best_thr(y, p):
    if y.sum() == 0 or len(y) < 10: return 0.5
    cands = np.unique(np.quantile(p, np.linspace(0.05, 0.99, 60)))
    return float(max(cands, key=lambda t: f1_score(y, p >= t, zero_division=0)))

def cv_eval(df, feats, k=5):
    X, y, g, t = df[feats].values, df.label.values, df.rid.values, df.task.values
    oof, pred = np.zeros(len(df)), np.zeros(len(df), int)
    for tr, te in GroupKFold(k).split(X, y, g):
        inner = np.zeros(len(tr))
        for a, b in GroupKFold(3).split(X[tr], y[tr], g[tr]):
            inner[b] = fit_lr(X[tr][a], y[tr][a]).predict_proba(X[tr][b])[:, 1]
        gthr = best_thr(y[tr], inner)
        thr = {tk: (best_thr(y[tr][t[tr] == tk], inner[t[tr] == tk]) if y[tr][t[tr] == tk].sum() >= 8 else gthr)
               for tk in set(t)}
        p = fit_lr(X[tr], y[tr]).predict_proba(X[te])[:, 1]
        oof[te] = p; pred[te] = [int(pi >= thr[tk]) for pi, tk in zip(p, t[te])]
    return oof, pred

def ece(y, p, bins=10):
    e, n = 0.0, len(y)
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (p >= lo) & (p < lo + 1 / bins + (1e-9 if lo + 1 / bins >= 1 else 0))
        if m.any(): e += m.sum() / n * abs(y[m].mean() - p[m].mean())
    return e

def boot_auc(y, p, g, B=200):
    r = np.random.default_rng(0); ug = np.unique(g); ix = {u: np.where(g == u)[0] for u in ug}; v = []
    for _ in range(B):
        idx = np.concatenate([ix[u] for u in r.choice(ug, len(ug))])
        if len(set(y[idx])) == 2: v.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(v, [2.5, 97.5]) if v else (np.nan, np.nan)

G = {"A": ["ent_mean", "ent_max", "nll_mean"], "B": ["se"], "D": ["dis"], "C": ["nosup"],
     "C3": ["doc_ent", "doc_con", "doc_neu"]}
FULL = "A+B+C+D  (full prototype)"
FULL3 = "A+B+C3+D  (full, 3-way doc check)"
VARIANTS = {
    "A  word-level doubt only":     G["A"],
    "B  semantic entropy only":     G["B"],
    "D  re-write disagreement only": G["D"],
    "C  document check only":       G["C"],
    "C3 3-way document check only": G["C3"],
    "A+B":                          G["A"] + G["B"],
    "A+C":                          G["A"] + G["C"],
    "A+C3":                         G["A"] + G["C3"],
    "B+C":                          G["B"] + G["C"],
    "A+B+C":                        G["A"] + G["B"] + G["C"],
    "A+B+C3":                       G["A"] + G["B"] + G["C3"],
    FULL:                           G["A"] + G["B"] + G["C"] + G["D"],
    FULL3:                          G["A"] + G["B"] + G["C3"] + G["D"],
}
OOF, PRED, table = {}, {}, []
y, g = df.label.values, df.rid.values
for name, feats in VARIANTS.items():
    oof, pred = cv_eval(df, feats); OOF[name], PRED[name] = oof, pred
    lo, hi = boot_auc(y, oof, g)
    table.append({"Method": name, "AUROC": roc_auc_score(y, oof), "AUROC 95% range": f"{lo:.2f}-{hi:.2f}",
                  "AUPRC": average_precision_score(y, oof), "Accuracy": accuracy_score(y, pred),
                  "Precision": precision_score(y, pred, zero_division=0), "Recall": recall_score(y, pred, zero_division=0),
                  "F1": f1_score(y, pred, zero_division=0), "ECE (lower=better)": ece(y, oof),
                  "Brier (lower=better)": brier_score_loss(y, oof)})
RES = pd.DataFrame(table).set_index("Method")
display(RES.round(3))
RES.round(4).to_csv(f"{WORK}/results_table.csv")
print("Random guessing would give AUROC 0.50 and AUPRC about", round(y.mean(), 3))
df["risk_full"] = OOF[FULL]; df["risk_cheap"] = OOF["A+C"]

1425 sentences from 200 answers | hallucinated sentences: 155 (10.9%)


task,Data2txt,QA,Summary
sentences,743,383,299
hallucinated,99,37,19


,AUROC,AUROC 95% range,AUPRC,Accuracy,Precision,Recall,F1,ECE (lower=better),Brier (lower=better)
Method,,,,,,,,,
A word-level doubt only,0.707,0.66-0.76,0.234,0.716,0.204,0.555,0.299,0.011,0.092
B semantic entropy only,0.591,0.55-0.63,0.134,0.526,0.139,0.645,0.228,0.005,0.096
D re-write disagreement only,0.665,0.62-0.71,0.164,0.601,0.164,0.652,0.262,0.002,0.094
C document check only,0.811,0.78-0.84,0.313,0.746,0.254,0.690,0.372,0.005,0.084
C3 3-way document check only,0.807,0.78-0.84,0.291,0.737,0.254,0.735,0.378,0.011,0.085
A+B,0.706,0.66-0.76,0.230,0.764,0.197,0.381,0.260,0.010,0.092
A+C,0.834,0.81-0.86,0.335,0.806,0.316,0.671,0.430,0.023,0.083
A+C3,0.831,0.80-0.86,0.328,0.806,0.307,0.619,0.410,0.018,0.083
B+C,0.822,0.80-0.85,0.356,0.798,0.290,0.594,0.390,0.016,0.082


Random guessing would give AUROC 0.50 and AUPRC about 0.109


In [10]:
# Per-task view (is it better on some tasks than others?)
rows = []
for tk, d in df.groupby("task"):
    if d.label.nunique() == 2:
        rows.append({"Task": tk, "Sentences": len(d), "Hallucinated": int(d.label.sum()),
                     "AUROC (full)": roc_auc_score(d.label, OOF[FULL][d.index]),
                     "AUROC (A only)": roc_auc_score(d.label, OOF["A  word-level doubt only"][d.index]),
                     "AUROC (B only)": roc_auc_score(d.label, OOF["B  semantic entropy only"][d.index]),
                     "AUROC (C only)": roc_auc_score(d.label, OOF["C  document check only"][d.index])})
TASKS = pd.DataFrame(rows).set_index("Task"); display(TASKS.round(3))
TASKS.round(4).to_csv(f"{WORK}/per_task.csv")

,Sentences,Hallucinated,AUROC (full),AUROC (A only),AUROC (B only),AUROC (C only)
Task,,,,,,
Data2txt,743,99,0.814,0.712,0.580,0.772
QA,383,37,0.881,0.722,0.611,0.856
Summary,299,19,0.839,0.766,0.625,0.763


### 6b. Does the expensive sampling really help? (paired test)
Each row compares two methods on the **same sentences**, resampling whole answers 500 times.
"Clearly different" means the 95% range of the difference does not include 0.

In [11]:
def paired_boot(sa, sb, B=500):
    r = np.random.default_rng(2); ug_ = np.unique(g); ix_ = {u: np.where(g == u)[0] for u in ug_}; da, dp = [], []
    for _ in range(B):
        idx = np.concatenate([ix_[u] for u in r.choice(ug_, len(ug_))])
        if len(set(y[idx])) < 2: continue
        da.append(roc_auc_score(y[idx], sa[idx]) - roc_auc_score(y[idx], sb[idx]))
        dp.append(average_precision_score(y[idx], sa[idx]) - average_precision_score(y[idx], sb[idx]))
    return da, dp

PAIRS = [
    ("Does the expensive sampling help?  FULL vs A+C", FULL, "A+C"),
    ("Does semantic entropy (B) add to A+C?  A+B+C vs A+C", "A+B+C", "A+C"),
    ("Does word doubt (A) add to the document check?  A+C vs C", "A+C", "C  document check only"),
    ("Document check vs word doubt  C vs A", "C  document check only", "A  word-level doubt only"),
    ("3-way vs plain document check inside the full model", FULL3, FULL),
]
def clear(d): return "YES" if (np.percentile(d, 2.5) > 0 or np.percentile(d, 97.5) < 0) else "no"
rows = []
for title, a_, b_ in PAIRS:
    da, dp = paired_boot(OOF[a_], OOF[b_])
    rows.append({"Question": title, "AUROC diff": np.mean(da),
                 "AUROC range": f"{np.percentile(da, 2.5):+.3f} to {np.percentile(da, 97.5):+.3f}", "AUROC clearly different?": clear(da),
                 "AUPRC diff": np.mean(dp), "AUPRC range": f"{np.percentile(dp, 2.5):+.3f} to {np.percentile(dp, 97.5):+.3f}",
                 "AUPRC clearly different?": clear(dp)})
PAIRED = pd.DataFrame(rows).set_index("Question"); display(PAIRED.round(3)); PAIRED.round(4).to_csv(f"{WORK}/paired_tests.csv")

,AUROC diff,AUROC range,AUROC clearly different?,AUPRC diff,AUPRC range,AUPRC clearly different?
Question,,,,,,
Does the expensive sampling help? FULL vs A+C,0.010,-0.004 to +0.023,no,0.029,+0.003 to +0.059,YES
Does semantic entropy (B) add to A+C? A+B+C vs A+C,0.001,-0.009 to +0.011,no,0.015,-0.004 to +0.035,no
Does word doubt (A) add to the document check? A+C vs C,0.023,+0.007 to +0.041,YES,0.022,-0.031 to +0.073,no
Document check vs word doubt C vs A,0.103,+0.043 to +0.165,YES,0.077,+0.015 to +0.151,YES
3-way vs plain document check inside the full model,-0.002,-0.006 to +0.001,no,-0.007,-0.028 to +0.012,no


### 6c. One global threshold or one threshold per task?
Same model, same folds. The only change is how the alarm cutoff is chosen: one cutoff for everything, or one per task type (QA / Summary / Data-to-text).
AUROC and calibration do not depend on the cutoff, so this compares precision, recall and F1.

In [12]:
def cv_pred(feats, per_task, k=5):
    X, yy, gg, tt = df[feats].values, df.label.values, df.rid.values, df.task.values
    pred = np.zeros(len(df), int)
    for tr, te in GroupKFold(k).split(X, yy, gg):
        inner = np.zeros(len(tr))
        for a, b in GroupKFold(3).split(X[tr], yy[tr], gg[tr]):
            inner[b] = fit_lr(X[tr][a], yy[tr][a]).predict_proba(X[tr][b])[:, 1]
        gthr = best_thr(yy[tr], inner)
        thr = {tk: (best_thr(yy[tr][tt[tr] == tk], inner[tt[tr] == tk]) if per_task and yy[tr][tt[tr] == tk].sum() >= 8 else gthr)
               for tk in set(tt)}
        p = fit_lr(X[tr], yy[tr]).predict_proba(X[te])[:, 1]
        pred[te] = [int(pi >= thr[tk]) for pi, tk in zip(p, tt[te])]
    return pred

FE = VARIANTS[FULL]
P_glob, P_task = cv_pred(FE, False), cv_pred(FE, True)
print("(check) per-task predictions match the earlier run:", bool(np.array_equal(P_task, PRED[FULL])))
rows = []
for nm, pr in [("one global threshold", P_glob), ("per-task thresholds", P_task)]:
    r_ = {"Method": nm, "Precision": precision_score(y, pr, zero_division=0), "Recall": recall_score(y, pr, zero_division=0),
          "F1": f1_score(y, pr, zero_division=0)}
    for tk in sorted(df.task.unique()):
        m = (df.task == tk).values
        r_[f"F1 {tk}"] = f1_score(y[m], pr[m], zero_division=0)
    rows.append(r_)
THRESH = pd.DataFrame(rows).set_index("Method"); display(THRESH.round(3))

rb = np.random.default_rng(3); ug_ = np.unique(g); ix_ = {u: np.where(g == u)[0] for u in ug_}; dd = []
for _ in range(500):
    idx = np.concatenate([ix_[u] for u in rb.choice(ug_, len(ug_))])
    if y[idx].sum() == 0: continue
    dd.append(f1_score(y[idx], P_task[idx], zero_division=0) - f1_score(y[idx], P_glob[idx], zero_division=0))
THR_RANGE = f"{np.percentile(dd, 2.5):+.3f} to {np.percentile(dd, 97.5):+.3f}"
print(f"F1 difference (per-task minus global): {np.mean(dd):+.3f}, 95% range {THR_RANGE} ->",
      "clearly different" if (np.percentile(dd, 2.5) > 0 or np.percentile(dd, 97.5) < 0) else "NOT clearly different")
THRESH.round(4).to_csv(f"{WORK}/threshold_comparison.csv")

(check) per-task predictions match the earlier run: True


,Precision,Recall,F1,F1 Data2txt,F1 QA,F1 Summary
Method,,,,,,
one global threshold,0.319,0.671,0.432,0.426,0.483,0.400
per-task thresholds,0.341,0.697,0.458,0.424,0.575,0.473


F1 difference (per-task minus global): +0.025, 95% range +0.010 to +0.044 -> clearly different


### Why the order of checks may matter: the four boxes
Is the model confident or unsure, and does the document back the sentence or not?
If many hallucinations sit in **"confident but not backed by the document"**, then word-level doubt alone will miss them (this is the Marie Curie case from the demo).

In [13]:
conf = (df.ent_mean <= df.ent_mean.median()).values
sup = (df.doc_ent >= 0.5).values
box = np.where(conf & sup, "1. model confident + document supports",
      np.where(conf & ~sup, "2. model confident + document does NOT support",
      np.where(~conf & sup, "3. model unsure + document supports", "4. model unsure + document does NOT support")))
BOX = (pd.DataFrame({"box": box, "label": y}).groupby("box").label.agg(["count", "sum", "mean"])
       .rename(columns={"count": "sentences", "sum": "hallucinated", "mean": "hallucination rate"}))
BOX["share of all hallucinations"] = BOX["hallucinated"] / max(y.sum(), 1)
display(BOX.round(3)); BOX.round(4).to_csv(f"{WORK}/four_boxes.csv")

,sentences,hallucinated,hallucination rate,share of all hallucinations
box,,,,
1. model confident + document supports,521,7,0.013,0.045
2. model confident + document does NOT support,192,36,0.188,0.232
3. model unsure + document supports,409,24,0.059,0.155
4. model unsure + document does NOT support,303,88,0.290,0.568


## 7. Three-way document check
Instead of only "supported or not", split into: **supported**, **contradicted**, **not mentioned**.
Which one matches the human marks best?

In [14]:
rows = []
for name, col, sign in [("document SUPPORTS it (low = risky)", "doc_ent", -1),
                        ("document CONTRADICTS it (high = risky)", "doc_con", 1),
                        ("document does NOT MENTION it (high = risky)", "doc_neu", 1)]:
    s = sign * df[col].values
    rows.append({"Signal": name, "AUROC alone": roc_auc_score(y, s), "AUPRC alone": average_precision_score(y, s),
                 "avg (hallucinated)": df.loc[y == 1, col].mean(), "avg (fine)": df.loc[y == 0, col].mean()})
THREE = pd.DataFrame(rows).set_index("Signal"); display(THREE.round(3))

cat = np.where(df.doc_ent >= 0.5, "supported", np.where(df.doc_con >= 0.5, "contradicted", "not mentioned"))
CAT = (pd.DataFrame({"category": cat, "label": y}).groupby("category").label.agg(["count", "sum", "mean"])
       .rename(columns={"count": "sentences", "sum": "hallucinated", "mean": "hallucination rate"}))
display(CAT.round(3))
print("Does the 3-way check beat the plain one when mixed in?  AUROC  A+B+C:", round(RES.loc['A+B+C', 'AUROC'], 3),
      "| A+B+C3:", round(RES.loc['A+B+C3', 'AUROC'], 3), "| C:", round(RES.loc['C  document check only', 'AUROC'], 3),
      "| C3:", round(RES.loc['C3 3-way document check only', 'AUROC'], 3))
THREE.round(4).to_csv(f"{WORK}/three_way.csv"); CAT.round(4).to_csv(f"{WORK}/three_way_categories.csv")

,AUROC alone,AUPRC alone,avg (hallucinated),avg (fine)
Signal,,,,
document SUPPORTS it (low = risky),0.815,0.314,0.261,0.674
document CONTRADICTS it (high = risky),0.602,0.144,0.203,0.133
document does NOT MENTION it (high = risky),0.778,0.307,0.611,0.285


,sentences,hallucinated,hallucination rate
category,,,
contradicted,84,26,0.310
not mentioned,411,98,0.238
supported,930,31,0.033


Does the 3-way check beat the plain one when mixed in?  AUROC  A+B+C: 0.835 | A+B+C3: 0.833 | C: 0.811 | C3: 0.807


## 8. THE MAIN TEST: which check should decide who gets the expensive re-writing?

Re-writing a sentence 5 times (signal B) is the expensive part. We only do it for the top **q%** of sentences. The rest are scored cheaply (word doubt + document check, `A+C`).

- **Doubt-first (the usual way):** the sentences with the highest word-level doubt get re-written.
- **Document-first (our idea):** the sentences the document does **not** support get re-written.
- **Random:** a fair comparison (average of 20 random picks).

Cost = share of sentences that get re-written. Quality = AUROC / AUPRC of the final score. At q = 100% all three are identical (everything is re-written).

In [15]:
QS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]
cheap, full = OOF["A+C"], OOF[FULL]
n = len(df); gen = df.gen_tok.values.astype(float)

def make_gate(score, q):
    k = int(round(q * n))
    if k >= n: return np.ones(n, bool)
    if k <= 0: return np.zeros(n, bool)
    thr = np.sort(score)[::-1][k - 1]
    return score >= thr

GATE_SCORES = {"doubt-first (word-level doubt)": df.ent_mean.values, "document-first (not backed by document)": df.nosup.values}
rows = []
for q in QS:
    for name, sc in GATE_SCORES.items():
        gt = make_gate(sc, q); h = np.where(gt, full, cheap)
        rows.append(dict(q=q, order=name, AUROC=roc_auc_score(y, h), AUPRC=average_precision_score(y, h),
                         caught=(gt & (y == 1)).sum() / max(y.sum(), 1), cost_sent=gt.mean(), cost_tok=gen[gt].sum() / gen.sum()))
    ra, rp, rc = [], [], []
    for sd in range(20):
        gt = make_gate(np.random.default_rng(sd).random(n), q); h = np.where(gt, full, cheap)
        ra.append(roc_auc_score(y, h)); rp.append(average_precision_score(y, h)); rc.append((gt & (y == 1)).sum() / max(y.sum(), 1))
    rows.append(dict(q=q, order="random pick", AUROC=np.mean(ra), AUPRC=np.mean(rp), caught=np.mean(rc), cost_sent=q, cost_tok=q))
GT = pd.DataFrame(rows)
for m in ["AUROC", "AUPRC", "caught"]:
    print("\n" + {"AUROC": "AUROC after gating (higher = better)", "AUPRC": "AUPRC after gating (higher = better)",
                  "caught": "share of hallucinated sentences that reach the expensive check"}[m])
    display(GT.pivot(index="q", columns="order", values=m).round(3))
GT.round(4).to_csv(f"{WORK}/gating_results.csv", index=False)

# is the difference between the two orders real? (resample whole answers, 300 times)
ug = np.unique(g); ix = {u: np.where(g == u)[0] for u in ug}; rb = np.random.default_rng(1)
gts = {q: {nm: make_gate(sc, q) for nm, sc in GATE_SCORES.items()} for q in QS[:-1]}
diffs = {q: [] for q in QS[:-1]}
for _ in range(300):
    idx = np.concatenate([ix[u] for u in rb.choice(ug, len(ug))])
    if len(set(y[idx])) < 2: continue
    for q in QS[:-1]:
        a = roc_auc_score(y[idx], np.where(gts[q]["document-first (not backed by document)"], full, cheap)[idx])
        b = roc_auc_score(y[idx], np.where(gts[q]["doubt-first (word-level doubt)"], full, cheap)[idx])
        diffs[q].append(a - b)
DIFF = pd.DataFrame({q: {"AUROC difference (document-first minus doubt-first)": np.mean(v),
                         "95% range low": np.percentile(v, 2.5), "95% range high": np.percentile(v, 97.5),
                         "range excludes 0?": bool(np.percentile(v, 2.5) > 0 or np.percentile(v, 97.5) < 0)}
                     for q, v in diffs.items() if v}).T
display(DIFF.round(3)); DIFF.round(4).to_csv(f"{WORK}/gating_difference.csv")
print("If the range includes 0, the two orders are NOT clearly different on this data - say so in the report.")

full_auc = roc_auc_score(y, full)
for name in GATE_SCORES:
    ok = [q for q in QS if GT[(GT.q == q) & (GT.order == name)].AUROC.iloc[0] >= full_auc - 0.01]
    print(f"{name}: smallest share of sentences that keeps AUROC within 0.01 of the full method = {min(ok):.0%}")


AUROC after gating (higher = better)


order,document-first (not backed by document),doubt-first (word-level doubt),random pick
q,,,
0.1,0.845,0.834,0.835
0.2,0.842,0.837,0.836
0.3,0.843,0.840,0.837
0.4,0.845,0.837,0.837
0.5,0.845,0.841,0.838
0.6,0.843,0.839,0.839
0.8,0.843,0.840,0.842
1.0,0.843,0.843,0.843



AUPRC after gating (higher = better)


order,document-first (not backed by document),doubt-first (word-level doubt),random pick
q,,,
0.1,0.371,0.340,0.338
0.2,0.361,0.343,0.342
0.3,0.361,0.351,0.345
0.4,0.362,0.346,0.347
0.5,0.363,0.352,0.350
0.6,0.362,0.354,0.352
0.8,0.362,0.361,0.357
1.0,0.362,0.362,0.362



share of hallucinated sentences that reach the expensive check


order,document-first (not backed by document),doubt-first (word-level doubt),random pick
q,,,
0.1,0.355,0.245,0.097
0.2,0.555,0.361,0.203
0.3,0.723,0.503,0.302
0.4,0.858,0.658,0.407
0.5,0.923,0.723,0.507
0.6,0.955,0.819,0.611
0.8,0.981,0.961,0.800
1.0,1.000,1.000,1.000


,AUROC difference (document-first minus doubt-first),95% range low,95% range high,range excludes 0?
0.1,0.010504,0.003551,0.019396,True
0.2,0.005168,-0.005281,0.015079,False
0.3,0.002054,-0.009337,0.011225,False
0.4,0.007344,-0.001621,0.016316,False
0.5,0.003507,-0.003235,0.010629,False
0.6,0.004816,-0.000119,0.010042,False
0.8,0.002655,-0.001068,0.006229,False


If the range includes 0, the two orders are NOT clearly different on this data - say so in the report.
doubt-first (word-level doubt): smallest share of sentences that keeps AUROC within 0.01 of the full method = 10%
document-first (not backed by document): smallest share of sentences that keeps AUROC within 0.01 of the full method = 10%


## 9. Early-warning test: does word-level doubt jump at the start of a wrong statement?
At each word we compare the doubt with normal text of the same window length. The alarm threshold is set so that normal text triggers it **10%** of the time.
If hallucinated spans trigger it much more often than 10% in their first few words, the paper's early-warning idea holds on real labels.

In [16]:
from numpy.lib.stride_tricks import sliding_window_view

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

K = 10
normal = {k: [] for k in range(1, K + 1)}; firsts = {k: [] for k in range(1, K + 1)}; span_list = []
for s in SPANS:
    ent = np.array(s["ent"], dtype=float); m = len(ent)
    inside = np.zeros(m, bool)
    for i0, i1 in s["spans"]: inside[i0:i1 + 1] = True
    for k in range(1, K + 1):
        if m >= k:
            mx = sliding_window_view(ent, k).max(1); bad = sliding_window_view(inside, k).any(1)
            normal[k] += mx[~bad].tolist()
    for i0, i1 in s["spans"]:
        span_list.append((ent, i0, i1))
        for k in range(1, K + 1):
            firsts[k].append(float(ent[i0:min(i0 + k, i1 + 1)].max()))

n_sp = len(span_list); rows = []; thr1 = None
for k in range(1, K + 1):
    if not normal[k] or n_sp == 0: continue
    thr = float(np.percentile(normal[k], 90)); thr1 = thr if k == 1 else thr1
    hit = int((np.array(firsts[k]) > thr).sum()); lo, hi = wilson(hit, n_sp)
    rows.append({"words after the start of the wrong span": k, "alarm fired (share of spans)": hit / n_sp,
                 "95% range": f"{lo:.0%}-{hi:.0%}", "normal text (by design)": 0.10})
EARLY = pd.DataFrame(rows).set_index("words after the start of the wrong span")
print(f"{n_sp} human-marked hallucinated spans")
display(EARLY.round(3)); EARLY.round(4).to_csv(f"{WORK}/early_warning.csv")

if thr1 is not None and n_sp:
    lags, lens = [], []
    for ent, i0, i1 in span_list:
        lens.append(i1 - i0 + 1)
        hit = [i - i0 for i in range(i0, i1 + 1) if ent[i] > thr1]
        if hit: lags.append(hit[0])
    chance = float(np.mean([1 - 0.9 ** L for L in lens]))
    print(f"Alarm fires somewhere inside the span for {len(lags)/n_sp:.0%} of spans "
          f"(if words were independent, chance level for spans this long would be about {chance:.0%}).")
    if lags: print(f"When it fires, the median delay is {np.median(lags):.0f} words after the span starts.")
    print("Average word-level doubt INSIDE wrong spans:",
          round(float(np.mean(np.concatenate([e[a:b + 1] for e, a, b in span_list]))), 3), "| in all other text:",
          round(float(np.mean(np.concatenate([np.delete(e, np.arange(a, b + 1)) for e, a, b in span_list]))), 3))

154 human-marked hallucinated spans


,alarm fired (share of spans),95% range,normal text (by design)
words after the start of the wrong span,,,
1,0.357,29%-44%,0.1
2,0.260,20%-33%,0.1
3,0.286,22%-36%,0.1
4,0.266,20%-34%,0.1
5,0.253,19%-33%,0.1
6,0.260,20%-33%,0.1
7,0.247,19%-32%,0.1
8,0.221,16%-29%,0.1
9,0.221,16%-29%,0.1


Alarm fires somewhere inside the span for 64% of spans (if words were independent, chance level for spans this long would be about 52%).
When it fires, the median delay is 0 words after the span starts.
Average word-level doubt INSIDE wrong spans: 0.967 | in all other text: 0.869


## 10. Our method vs the paper's baseline (answer level) and the cost
The baseline gives one score per answer, so we turn our sentence scores into one answer score by taking the **highest-risk sentence**.
Cost is counted as "token slots generated by the GPU for re-writing" per answer.

In [17]:
if not BASE:
    print("Baseline was not run (RUN_BASELINE = False or BASE_N too small).")
else:
    B = pd.DataFrame(BASE); B["rid"] = B["rid"].astype(str)
    agg = (df.groupby("rid").agg(max_risk_full=("risk_full", "max"), max_risk_cheap=("risk_cheap", "max"),
                                 max_nosup=("nosup", "max"), max_doubt=("ent_mean", "max"), max_se=("se", "max"))
           .reset_index().merge(B[["rid", "se_whole", "label", "gen_tok"]], on="rid"))
    ya = agg.label.values; ga = agg.rid.values
    print(f"{len(agg)} answers in the comparison ({int(ya.sum())} with a human-marked hallucination)")
    if len(set(ya)) < 2:
        print("Need both clean and hallucinated answers - raise BASE_N.")
    else:
        names = {"se_whole": "BASELINE: whole-answer semantic entropy (the paper)", "max_doubt": "word-level doubt (A), worst sentence",
                 "max_se": "sentence-level semantic entropy (B), worst sentence", "max_nosup": "document check (C), worst sentence",
                 "max_risk_cheap": "A+C combined, worst sentence", "max_risk_full": "FULL prototype, worst sentence"}
        rows = []
        for c, nm in names.items():
            lo, hi = boot_auc(ya, agg[c].values, ga, B=200)
            rows.append({"Method (answer level)": nm, "AUROC": roc_auc_score(ya, agg[c].values), "95% range": f"{lo:.2f}-{hi:.2f}",
                         "AUPRC": average_precision_score(ya, agg[c].values)})
        ANS = pd.DataFrame(rows).set_index("Method (answer level)"); display(ANS.round(3)); ANS.round(4).to_csv(f"{WORK}/answer_level_vs_baseline.csv")

        Q_REPORT = 0.4
        n_ans = df.rid.nunique()
        c_base = float(B.gen_tok.mean())
        c_full = float(gen.sum() / n_ans)
        c_gate = float(gen[make_gate(df.nosup.values, Q_REPORT)].sum() / n_ans)
        COST = pd.DataFrame({"token slots generated per answer": [c_base, c_full, c_gate]},
                            index=["BASELINE: 5 whole-answer re-writes", "ours: re-write every sentence",
                                   f"ours: document-first, only {Q_REPORT:.0%} of sentences"])
        COST["relative to baseline"] = COST.iloc[:, 0] / c_base
        display(COST.round(2)); COST.round(3).to_csv(f"{WORK}/cost_vs_baseline.csv")
        print("Note: the baseline only gives one score per answer and cannot point to the wrong sentence.")

100 answers in the comparison (51 with a human-marked hallucination)


,AUROC,95% range,AUPRC
Method (answer level),,,
BASELINE: whole-answer semantic entropy (the paper),0.472,0.37-0.58,0.485
"word-level doubt (A), worst sentence",0.786,0.70-0.86,0.760
"sentence-level semantic entropy (B), worst sentence",0.554,0.49-0.61,0.538
"document check (C), worst sentence",0.740,0.63-0.84,0.666
"A+C combined, worst sentence",0.798,0.70-0.88,0.725
"FULL prototype, worst sentence",0.826,0.74-0.90,0.751


,token slots generated per answer,relative to baseline
BASELINE: 5 whole-answer re-writes,808.15,1.00
ours: re-write every sentence,1321.95,1.64
"ours: document-first, only 40% of sentences",494.72,0.61


Note: the baseline only gives one score per answer and cannot point to the wrong sentence.


### 10b. The fair baseline: SelfCheck-style check against our signals
Same answers for every row (the ones that have SelfCheck-style scores), so the comparison is fair.
Sentence level first, then answer level (worst sentence), then cost.

In [18]:
if not SC:
    print("SelfCheck-style scores are not available (RUN_BASELINE = False or section 5b not run).")
else:
    S = pd.DataFrame(SC); S["rid"] = S["rid"].astype(str)
    d2 = df.reset_index().merge(S[["rid", "k", "sc", "gen_tok"]].rename(columns={"gen_tok": "sc_cost"}), on=["rid", "k"], how="inner")
    ix2 = d2["index"].values; ys = d2.label.values; gs = d2.rid.values
    print(f"{len(d2)} sentences from {d2.rid.nunique()} answers ({int(ys.sum())} hallucinated sentences)")
    cols = {"SelfCheck-style (re-writes contradict the written sentence)": d2.sc.values,
            "A  word-level doubt": d2.ent_mean.values, "B  semantic entropy of re-written sentence": d2.se.values,
            "C  document check": d2.nosup.values, "A+C combined": OOF["A+C"][ix2], "FULL prototype": OOF[FULL][ix2]}
    rows = []
    for nm, sc_ in cols.items():
        if len(set(ys)) < 2: break
        lo, hi = boot_auc(ys, sc_, gs, B=200)
        rows.append({"Method (sentence level)": nm, "AUROC": roc_auc_score(ys, sc_), "95% range": f"{lo:.2f}-{hi:.2f}",
                     "AUPRC": average_precision_score(ys, sc_)})
    SC_SENT = pd.DataFrame(rows).set_index("Method (sentence level)"); display(SC_SENT.round(3)); SC_SENT.round(4).to_csv(f"{WORK}/selfcheck_sentence_level.csv")

    a2 = d2.groupby("rid").agg(sc_max=("sc", "max"), sc_mean=("sc", "mean"), doubt=("ent_mean", "max"), doc=("nosup", "max"),
                               cheap=("risk_cheap", "max"), full=("risk_full", "max"), sc_cost=("sc_cost", "first")).reset_index()
    a2["label"] = [int(bool(by_rid[r_]["labels"])) for r_ in a2.rid]
    if BASE:
        Bb = pd.DataFrame(BASE); Bb["rid"] = Bb["rid"].astype(str); a2 = a2.merge(Bb[["rid", "se_whole"]], on="rid", how="left")
    ya = a2.label.values
    names = {"sc_max": "SelfCheck-style, worst sentence", "sc_mean": "SelfCheck-style, average sentence", "se_whole": "whole-answer semantic entropy (the paper)",
             "doubt": "word-level doubt (A), worst sentence", "doc": "document check (C), worst sentence",
             "cheap": "A+C combined, worst sentence", "full": "FULL prototype, worst sentence"}
    rows = []
    for c, nm in names.items():
        if c not in a2 or a2[c].isna().any() or len(set(ya)) < 2: continue
        lo, hi = boot_auc(ya, a2[c].values, a2.rid.values, B=200)
        rows.append({"Method (answer level)": nm, "AUROC": roc_auc_score(ya, a2[c].values), "95% range": f"{lo:.2f}-{hi:.2f}",
                     "AUPRC": average_precision_score(ya, a2[c].values)})
    SC_ANS = pd.DataFrame(rows).set_index("Method (answer level)")
    print(f"\nAnswer level: {len(a2)} answers ({int(ya.sum())} with a hallucination)")
    display(SC_ANS.round(3)); SC_ANS.round(4).to_csv(f"{WORK}/selfcheck_answer_level.csv")
    print(f"Cost: SelfCheck-style = {a2.sc_cost.mean():.0f} token slots per answer (5 whole-answer re-writes), "
          f"same as the whole-answer baseline. Ours with document-first gating at 40% = {gen[make_gate(df.nosup.values, 0.4)].sum() / df.rid.nunique():.0f}.")

713 sentences from 100 answers (72 hallucinated sentences)


,AUROC,95% range,AUPRC
Method (sentence level),,,
SelfCheck-style (re-writes contradict the written sentence),0.666,0.61-0.71,0.152
A word-level doubt,0.691,0.61-0.75,0.202
B semantic entropy of re-written sentence,0.621,0.57-0.67,0.133
C document check,0.812,0.77-0.86,0.266
A+C combined,0.840,0.81-0.88,0.297
FULL prototype,0.853,0.82-0.89,0.325



Answer level: 100 answers (51 with a hallucination)


,AUROC,95% range,AUPRC
Method (answer level),,,
"SelfCheck-style, worst sentence",0.564,0.45-0.66,0.582
"SelfCheck-style, average sentence",0.479,0.37-0.59,0.489
whole-answer semantic entropy (the paper),0.472,0.37-0.58,0.485
"word-level doubt (A), worst sentence",0.786,0.70-0.86,0.760
"document check (C), worst sentence",0.740,0.63-0.84,0.666
"A+C combined, worst sentence",0.798,0.70-0.88,0.725
"FULL prototype, worst sentence",0.826,0.74-0.90,0.751


Cost: SelfCheck-style = 775 token slots per answer (5 whole-answer re-writes), same as the whole-answer baseline. Ours with document-first gating at 40% = 495.


## 11. Safety rule for the demo (the Marie Curie problem)
Rule: if the document check says a sentence is **almost surely not backed** (`not backed > S`), never show it green - show at least amber.
Below: what this rule does to precision and recall on the real labels. Do not tune `S` too hard on the same data you report.

In [19]:
base_pred = PRED[FULL]; rows = [{"rule": "none (model only)", "Precision": precision_score(y, base_pred, zero_division=0),
                                  "Recall": recall_score(y, base_pred, zero_division=0), "F1": f1_score(y, base_pred, zero_division=0),
                                  "flagged sentences": int(base_pred.sum())}]
for S in [0.8, 0.9, 0.95, 0.99]:
    p2 = (base_pred.astype(bool) | (df.nosup.values > S)).astype(int)
    rows.append({"rule": f"also flag if not backed > {S}", "Precision": precision_score(y, p2, zero_division=0),
                 "Recall": recall_score(y, p2, zero_division=0), "F1": f1_score(y, p2, zero_division=0), "flagged sentences": int(p2.sum())})
SAFE = pd.DataFrame(rows).set_index("rule"); display(SAFE.round(3)); SAFE.round(4).to_csv(f"{WORK}/safety_rule.csv")

,Precision,Recall,F1,flagged sentences
rule,,,,
none (model only),0.341,0.697,0.458,317
also flag if not backed > 0.8,0.297,0.748,0.425,391
also flag if not backed > 0.9,0.309,0.729,0.434,366
also flag if not backed > 0.95,0.315,0.716,0.438,352
also flag if not backed > 0.99,0.325,0.697,0.444,332


## 12. One-screen summary to copy into your review

In [20]:
def g_(df_, col, default=np.nan):
    try: return df_.loc[col]
    except Exception: return default

print("SUMMARY FOR REVIEW"); print("------------------")
print(f"Data: {df.rid.nunique()} answers, {len(df)} sentences, {df.label.sum()} hallucinated ({df.label.mean():.1%}).")
print(f"Word-level doubt alone      : AUROC {RES.loc['A  word-level doubt only','AUROC']:.3f}")
print(f"Semantic entropy alone      : AUROC {RES.loc['B  semantic entropy only','AUROC']:.3f}")
print(f"Document check alone        : AUROC {RES.loc['C  document check only','AUROC']:.3f}   (3-way: {RES.loc['C3 3-way document check only','AUROC']:.3f})")
print(f"Full prototype              : AUROC {RES.loc[FULL,'AUROC']:.3f}, F1 {RES.loc[FULL,'F1']:.3f}, ECE {RES.loc[FULL,'ECE (lower=better)']:.3f}, Brier {RES.loc[FULL,'Brier (lower=better)']:.3f}")
print(f"Best variant overall        : {RES['AUROC'].idxmax()}")
for q in [0.3, 0.4]:
    a = GT[(GT.q == q) & (GT.order.str.startswith("document"))].AUROC.iloc[0]
    b = GT[(GT.q == q) & (GT.order.str.startswith("doubt"))].AUROC.iloc[0]
    r_ = GT[(GT.q == q) & (GT.order == "random pick")].AUROC.iloc[0]
    print(f"Gating at {q:.0%} of sentences : document-first AUROC {a:.3f} | doubt-first {b:.3f} | random {r_:.3f}")
if len(EARLY): print(f"Early alarm in first 5 words of a wrong span: {EARLY.iloc[min(4, len(EARLY)-1)]['alarm fired (share of spans)']:.0%} (normal text: 10%)")
if BASE and 'ANS' in globals():
    print(f"Answer level: baseline AUROC {ANS.iloc[0]['AUROC']:.3f} | our full prototype {ANS.iloc[-1]['AUROC']:.3f}")
if 'PAIRED' in globals():
    q_ = PAIRED.iloc[0]
    print(f"Does the expensive sampling help (FULL vs A+C)? AUROC diff {q_['AUROC diff']:+.3f} ({q_['AUROC range']}) -> clearly different: {q_['AUROC clearly different?']}")
if 'THRESH' in globals():
    print(f"Thresholds: global F1 {THRESH.loc['one global threshold','F1']:.3f} | per-task F1 {THRESH.loc['per-task thresholds','F1']:.3f} (difference range {THR_RANGE})")
if 'SC_SENT' in globals() and len(SC_SENT):
    print(f"Fair baseline (SelfCheck-style), sentence level: AUROC {SC_SENT.iloc[0]['AUROC']:.3f} vs our full prototype on the same sentences {SC_SENT.loc['FULL prototype','AUROC']:.3f}")
print("\nRemember: say 'not clearly different' whenever the 95% ranges overlap.")

SUMMARY FOR REVIEW
------------------
Data: 200 answers, 1425 sentences, 155 hallucinated (10.9%).
Word-level doubt alone      : AUROC 0.707
Semantic entropy alone      : AUROC 0.591
Document check alone        : AUROC 0.811   (3-way: 0.807)
Full prototype              : AUROC 0.843, F1 0.458, ECE 0.014, Brier 0.080
Best variant overall        : A+B+C+D  (full prototype)
Gating at 30% of sentences : document-first AUROC 0.843 | doubt-first 0.840 | random 0.837
Gating at 40% of sentences : document-first AUROC 0.845 | doubt-first 0.837 | random 0.837
Early alarm in first 5 words of a wrong span: 25% (normal text: 10%)
Answer level: baseline AUROC 0.472 | our full prototype 0.826
Does the expensive sampling help (FULL vs A+C)? AUROC diff +0.010 (-0.004 to +0.023) -> clearly different: no
Thresholds: global F1 0.432 | per-task F1 0.458 (difference range +0.010 to +0.044)
Fair baseline (SelfCheck-style), sentence level: AUROC 0.666 vs our full prototype on the same sentences 0.853

Remembe

## 13. Live demo
Paste a document and a question. The model writes an answer (or paste an answer to check), and every sentence is coloured:
**green = looks fine, amber = double-check, red = likely hallucination.** Hover a sentence to see its signals.
The safety rule makes sure a sentence the document clearly does not back is never green.

In [21]:
import gradio as gr

SAFETY = 0.9
FEATS = G["A"] + G["B"] + G["C"] + G["D"]
FINAL = fit_lr(df[FEATS].values, y)
full_oof = OOF[FULL]
GTHR = best_thr(y, full_oof)
THR = {tk: (best_thr(y[(df.task == tk).values], full_oof[(df.task == tk).values])
            if y[(df.task == tk).values].sum() >= 8 else GTHR) for tk in df.task.unique()}
print("warning thresholds by task:", {k: round(v, 3) for k, v in THR.items()}, "| fallback:", round(GTHR, 3))

@torch.no_grad()
def write_answer(prompt):
    ids = tok(chat_prompt(prompt), return_tensors="pt").to(dev)
    out = model.generate(**ids, do_sample=False, temperature=None, top_p=None, top_k=None,
                         max_new_tokens=160, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def analyze(task, question, document, given_answer):
    if not document.strip(): return "<p>Please paste a document.</p>", None
    if task == "QA":
        prompt = (f"Briefly answer the following question: {question}\n"
                  f"Bear in mind that your response should be strictly based on the following passages:\n{document}")
    else:
        prompt = f"Summarize the following text within 100 words:\n{document}"
    answer = given_answer.strip() or write_answer(prompt)
    out = score_response(prompt, answer, document, labels=None)
    if out is None: return "<p>Could not score this (text too long or too short).</p>", None
    rows, _ = out
    p = FINAL.predict_proba(pd.DataFrame(rows)[FEATS].values)[:, 1]
    thr = THR.get(task, GTHR)
    parts = []
    for r, pi in zip(rows, p):
        unbacked = r["nosup"] > SAFETY
        colour = "#f8b4b4" if pi >= thr else ("#fde68a" if (pi >= 0.6 * thr or unbacked) else "#bbf7d0")
        tip = (f"risk {pi:.2f} | word doubt {r['ent_mean']:.2f} | meaning doubt {r['se']:.2f} | "
               f"not backed by document {r['nosup']:.2f} | contradicted {r['doc_con']:.2f}")
        parts.append(f"<span title='{tip}' style='background:{colour};padding:2px 4px;border-radius:4px;margin-right:2px'>{html.escape(r['text'])}</span>")
    legend = "<p style='font-size:13px'>green = looks fine &nbsp; amber = double-check &nbsp; red = likely hallucination</p>"
    table = pd.DataFrame({"sentence": [r["text"][:90] for r in rows], "risk": np.round(p, 2),
                          "word doubt": [round(r["ent_mean"], 2) for r in rows],
                          "meaning doubt": [round(r["se"], 2) for r in rows],
                          "not backed by document": [round(r["nosup"], 2) for r in rows],
                          "contradicted": [round(r["doc_con"], 2) for r in rows]})
    return legend + "<div style='line-height:2.1;color:#111'>" + " ".join(parts) + "</div>", table

demo = gr.Interface(
    fn=analyze,
    inputs=[gr.Radio(["QA", "Summary"], value="QA", label="Task type"),
            gr.Textbox(label="Question (for QA)", value="Where was Marie Curie born, and which university did she teach at?"),
            gr.Textbox(lines=8, label="Document",
                       value="Marie Curie was born in Warsaw in 1867. She won the Nobel Prize in Physics in 1903 and the Nobel Prize in Chemistry in 1911."),
            gr.Textbox(lines=4, label="Answer to check (leave empty to let the model write one)")],
    outputs=[gr.HTML(label="Result"), gr.Dataframe(label="Signals per sentence")],
    title="Hallucination Early-Warning Prototype",
    description="Sentences are coloured by how likely they are to be hallucinated. Uses word-level doubt, meaning-level doubt (semantic entropy) and a document check.")
demo.launch(share=True)

warning thresholds by task: {'Data2txt': 0.222, 'QA': 0.142, 'Summary': 0.314} | fallback: 0.223
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: [temporary public link removed]

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 14. Save your results
Everything is already saved in your `WORK` folder (Google Drive if `USE_DRIVE = True`). This cell also downloads the main files.

In [22]:
df.to_csv(f"{WORK}/sentences_scored.csv", index=False)
try:
    from google.colab import files
    for f in ["results_table.csv", "gating_results.csv", "gating_difference.csv", "early_warning.csv", "sentences_scored.csv"]:
        if os.path.exists(f"{WORK}/{f}"): files.download(f"{WORK}/{f}")
except Exception as e:
    print("Download skipped:", e, "| files are in", WORK)